<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/07.Training_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 7 — Training Pipeline



Model bahasa tidak belajar dari teks secara langsung. Teks harus diubah menjadi angka, disusun menjadi batch, dilewatkan ke model, dibandingkan dengan jawaban yang benar, lalu selisihnya dipakai untuk menggeser bobot sedikit demi sedikit. Rangkaian langkah itulah yang disebut **training pipeline**, dan bab ini membangunnya dari ujung ke ujung.

Buku menyusun pipeline dari tujuh komponen: pembuatan dataset, arsitektur model, loop training, optimasi, evaluasi, checkpointing, dan logging. Contohnya adalah *fine-tuning* **GPT-2 124M** (124 juta parameter) pada Wikipedia. Kita mengikuti skenario yang sama dalam skala yang muat di satu GPU Colab: beberapa ratus artikel Wikipedia, kurang dari 300 langkah update, dan satu set artikel *held-out* (tidak pernah dilihat saat training) untuk mengukur apakah model benar-benar membaik.

Beberapa listing buku tidak bisa dijalankan apa adanya. Listing itu ditampilkan di kotak kutipan pendek beserta penyebabnya, dan versi yang berjalan langsung mengikutinya. Di sepanjang jalan ada empat pertanyaan *under the hood* yang dijawab dengan pengukuran:

1. Seberapa mahal kesalahan menggeser label secara manual?
2. Seberapa sering gradient clipping benar-benar memotong?
3. Bagaimana batch size menukar memori dengan throughput?
4. Kapan akumulasi gradien benar-benar setara dengan satu batch besar?

> **Status eksekusi.** Output notebook sengaja dikosongkan, jadi setiap angka yang muncul adalah hasil run-mu sendiri. Profil default (`gpu`) **belum pernah dijalankan di Colab atau CUDA**. Yang sudah diuji adalah jalur kode yang sama dalam profil `cpu_smoke` di CPU lokal (plus satu run di GPU Apple/MPS untuk mengecek konsistensi device), memakai GPT-2 mini berbobot acak dan data kecil, serta bagian data profil `gpu` (streaming dan tokenisasi, tanpa model) di CPU. Angka pembanding di narasi selalu diberi label sumbernya: run historis notebook versi lama (CPU, model mini), validasi lokal bagian data, atau aritmetika yang bisa dihitung ulang.

## Persiapan: pastikan GPU benar-benar bekerja

Fine-tuning GPT-2 124M di CPU bisa memakan waktu berjam-jam, jadi notebook ini berhenti sejak awal kalau GPU tidak tersedia, bukan diam-diam pindah ke CPU. Sel pertama menentukan mode run dan folder kerja. Semua cache Hugging Face, log, dan checkpoint ditulis ke `ch07_work/`. Weights & Biases dimatikan (`WANDB_MODE=disabled`) dan tidak ada login apa pun; log disimpan lokal sebagai file JSONL dan event TensorBoard.

Mode `cpu_smoke` hanya untuk menguji bahwa kode berjalan: GPT-2 mini berbobot acak, puluhan artikel, dan langkah training satu digit. Mode itu harus dipilih secara eksplisit, dengan mengubah `RUN_MODE` di sel berikut atau menyetel variabel lingkungan `CH07_RUN_MODE=cpu_smoke`.

In [1]:
import importlib.util
import os
import subprocess
import sys
import time
from pathlib import Path

# "gpu" = profil default bab ini. "cpu_smoke" = opt-in eksplisit untuk uji jalannya kode (model mini, data kecil).
RUN_MODE = os.environ.get("CH07_RUN_MODE", "gpu")

WORK_DIR = Path(os.environ.get("CH07_WORK_DIR", "ch07_work")).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d-%H%M%S")

# Cache Hugging Face ikut di WORK_DIR; telemetri hosted dan wandb dimatikan.
os.environ.setdefault("HF_HOME", str(WORK_DIR / "hf_home"))
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["WANDB_MODE"] = "disabled"

# Colab sudah menyediakan paket ini; instal hanya yang benar-benar belum ada.
REQUIRED = {"torch": "torch", "transformers": "transformers", "datasets": "datasets", "tensorboard": "tensorboard"}
missing = [pip_name for module, pip_name in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("menginstal:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("RUN_MODE :", RUN_MODE)
print("WORK_DIR :", WORK_DIR)
print("RUN_ID   :", RUN_ID)

RUN_MODE : gpu
WORK_DIR : /content/ch07_work
RUN_ID   : 20261029-032535


Gate di bawah tidak cukup mengecek `torch.cuda.is_available()`. Ia mengalokasikan matriks 2048×2048 di GPU, mengalikannya, menunggu GPU selesai (`synchronize`), lalu memeriksa bahwa hasilnya finite. Kalau driver atau runtime bermasalah, kegagalannya muncul di sini, bukan di tengah training.

Sel yang sama memilih tipe data untuk *mixed precision* yang dipakai di bagian scaling. GPU dengan compute capability 8.0 ke atas (misalnya A100 atau L4) mendukung `bfloat16` secara native. T4 (7.5) tidak, sehingga jalurnya `float16` ditambah `GradScaler`.

In [2]:
import torch

if RUN_MODE == "gpu":
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA tidak terdeteksi. Di Colab: Runtime > Change runtime type > pilih GPU (misalnya T4), "
            "lalu jalankan ulang dari sel pertama. Untuk sekadar menguji kode tanpa GPU, set "
            "RUN_MODE = 'cpu_smoke' secara eksplisit."
        )
    DEVICE = torch.device("cuda")
    probe = torch.randn(2048, 2048, device=DEVICE)
    probe_out = probe @ probe.T
    torch.cuda.synchronize()
    if not torch.isfinite(probe_out).all():
        raise RuntimeError("Gate CUDA gagal: hasil matmul uji tidak finite.")
    props = torch.cuda.get_device_properties(DEVICE)
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    # Compute capability >= 8.0 punya bfloat16 native; T4 (7.5) memakai float16 + GradScaler.
    AMP_DTYPE = torch.bfloat16 if props.major >= 8 else torch.float16
    del probe, probe_out
    torch.cuda.empty_cache()
    print(f"GPU: {props.name} | compute capability {props.major}.{props.minor} | "
          f"memori bebas {free_bytes / 2**30:.1f} / {total_bytes / 2**30:.1f} GiB")
elif RUN_MODE == "cpu_smoke":
    DEVICE = torch.device("cpu")
    AMP_DTYPE = None
    print("MODE cpu_smoke (opt-in): GPT-2 mini berbobot acak dan data kecil. "
          "Angka di mode ini hanya menguji jalannya kode, bukan hasil bab.")
else:
    raise ValueError(f"RUN_MODE tidak dikenal: {RUN_MODE!r}")

PIN_MEMORY = DEVICE.type == "cuda"
print("DEVICE:", DEVICE, "| dtype autocast untuk bagian scaling:", AMP_DTYPE)

GPU: Tesla T4 | compute capability 7.5 | memori bebas 14.3 / 14.6 GiB
DEVICE: cuda | dtype autocast untuk bagian scaling: torch.float16


Semua knob skala dikumpulkan di satu profil supaya mudah dibandingkan dengan buku. Buku memakai seluruh Wikipedia (sekitar 6,4 juta artikel) ditambah BookCorpus, konteks 512–1024 token, dan 3 epoch. Profil `gpu` memakai 800 artikel training, 100 artikel held-out, konteks 512, dan 1 epoch. Yang dipertahankan dari buku adalah modelnya (GPT-2 124M pretrained), learning rate 5e-5, batch 8, dan bentuk pipeline-nya.

In [3]:
import gc
import itertools
import json
import math
from types import SimpleNamespace

import datasets
import numpy as np
import torch.nn.functional as F
import transformers
from datasets import Dataset as HFDataset
from datasets import concatenate_datasets, load_dataset
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GPT2Config,
    GPT2LMHeadModel,
    get_linear_schedule_with_warmup,
)

transformers.logging.set_verbosity_error()
datasets.utils.logging.set_verbosity_error()
datasets.disable_progress_bars()

PROFILES = {
    "gpu": dict(
        n_train_articles=800, n_val_articles=100, seq_len=512,
        batch_size=8, num_epochs=1, warmup_steps=10, lr=5e-5,
        opt_steps=100, trainer_batches_per_epoch=30, trainer_log_interval=10,
        micro_batch=4, accum_steps=4, n_micro_batches=30,
        val_blocks=128, eval_batch=8, sweep_batches=(1, 2, 4, 8),
    ),
    "cpu_smoke": dict(
        n_train_articles=24, n_val_articles=8, seq_len=64,
        batch_size=4, num_epochs=1, warmup_steps=2, lr=5e-4,
        opt_steps=6, trainer_batches_per_epoch=2, trainer_log_interval=1,
        micro_batch=2, accum_steps=4, n_micro_batches=6,
        val_blocks=8, eval_batch=4, sweep_batches=(1, 2),
    ),
}
CFG = SimpleNamespace(**PROFILES[RUN_MODE])
SEQ_LEN = CFG.seq_len
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | "
      f"transformers {transformers.__version__} | datasets {datasets.__version__}")
print("profil", RUN_MODE, vars(CFG))

python 3.13.16 | torch 2.11.0+cu130 | transformers 5.20.2 | datasets 4.8.5
profil gpu {'n_train_articles': 800, 'n_val_articles': 100, 'seq_len': 512, 'batch_size': 8, 'num_epochs': 1, 'warmup_steps': 10, 'lr': 5e-05, 'opt_steps': 100, 'trainer_batches_per_epoch': 30, 'trainer_log_interval': 10, 'micro_batch': 4, 'accum_steps': 4, 'n_micro_batches': 30, 'val_blocks': 128, 'eval_batch': 8, 'sweep_batches': (1, 2, 4, 8)}


Satu fungsi bantu dipakai di banyak tempat, jadi perlu dipahami dulu: `causal_lm_nll`. Model bahasa *causal* memprediksi token berikutnya, sehingga logit di posisi *t* dibandingkan dengan token di posisi *t+1*. Fungsi ini melakukan pergeseran satu posisi itu, melewati label bernilai `-100`, lalu mengembalikan **jumlah** cross-entropy beserta jumlah token yang dihitung. Dengan menyimpan jumlah, bukan rata-rata, kita bisa merata-ratakan per token secara tepat di seluruh dataset atau di seluruh langkah akumulasi.

`evaluate_lm` memakai fungsi itu pada data held-out. Hasilnya adalah **loss** (rata-rata cross-entropy per token, satuannya nat) dan **perplexity** = e^loss. Perplexity 30 berarti, rata-rata, model sebingung orang yang harus memilih acak di antara 30 token.

In [4]:
def to_device(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def peak_mem_text():
    if DEVICE.type != "cuda":
        return f"n/a ({DEVICE.type})"
    return f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GiB"


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def require_finite(value, what):
    if not torch.isfinite(value).all():
        raise FloatingPointError(f"{what} tidak finite: {value}")


def causal_lm_nll(logits, labels):
    """Jumlah cross-entropy causal LM dan jumlah token yang dihitung.

    Logit posisi t dipasangkan dengan label posisi t+1; label -100 dilewati.
    """
    shift_logits = logits[:, :-1, :].float()
    shift_labels = labels[:, 1:]
    nll_sum = F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
        ignore_index=-100,
        reduction="sum",
    )
    n_tokens = (shift_labels != -100).sum()
    return nll_sum, n_tokens


@torch.no_grad()
def evaluate_lm(model, dataloader):
    """Loss rata-rata per token (nat) dan perplexity pada dataloader berlabel."""
    assert next(model.parameters()).device.type == DEVICE.type
    was_training = model.training
    model.eval()
    nll_total = torch.zeros((), dtype=torch.float64, device=DEVICE)
    tok_total = torch.zeros((), dtype=torch.long, device=DEVICE)
    for batch in dataloader:
        batch = to_device(batch)
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        nll_sum, n_tok = causal_lm_nll(logits, batch["labels"])
        nll_total += nll_sum.double()
        tok_total += n_tok
    model.train(was_training)
    loss = (nll_total / tok_total).item()
    if not math.isfinite(loss):
        raise FloatingPointError(f"loss evaluasi tidak finite: {loss}")
    return {"loss": loss, "ppl": math.exp(loss), "tokens": int(tok_total.item())}


def pad_rows(seqs, label_seqs):
    """Susun list token menjadi batch ber-padding kanan; label di posisi padding = -100."""
    n = len(seqs)
    input_ids = torch.full((n, SEQ_LEN), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((n, SEQ_LEN), dtype=torch.long)
    labels = torch.full((n, SEQ_LEN), -100, dtype=torch.long)
    for row, (ids, lab) in enumerate(zip(seqs, label_seqs)):
        input_ids[row, :len(ids)] = torch.tensor(ids)
        attention_mask[row, :len(ids)] = 1
        labels[row, :len(lab)] = torch.tensor(lab)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def rows_loader(rows, batch_size):
    items = [{k: v[i] for k, v in rows.items()} for i in range(rows["input_ids"].size(0))]
    return DataLoader(items, batch_size=batch_size, pin_memory=PIN_MEMORY)


class LocalLogger:
    """Pengganti wandb.log: satu baris JSON per kejadian, ditulis ke file lokal."""

    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, **metrics):
        with open(self.path, "a") as f:
            f.write(json.dumps(metrics) + "\n")


metrics_log = LocalLogger(WORK_DIR / "logs" / f"metrics_{RUN_ID}.jsonl")
print("log metrik lokal:", metrics_log.path)

log metrik lokal: /content/ch07_work/logs/metrics_20261029-032535.jsonl


> **Listing buku: blok import yang perlu diperbaiki**
>
> ```python
> from transformers import (
>     AutoTokenizer, AutoModelForCausalLM, AdamW,
>     get_linear_schedule_with_warmup
> from datasets import load_dataset
> ```
>
> Kurung di baris kedua tidak pernah ditutup, sehingga Python menolak seluruh sel dengan `SyntaxError: '(' was never closed`. Setelah kurung ditutup pun, `transformers` modern tidak lagi menyediakan `AdamW` (`ImportError`). Optimizer itu tinggal di `torch.optim.AdamW`, import yang juga dipakai buku sendiri di bagian optimasi. Sel import di atas sudah memakai versi yang benar.

## 1. Ingestion: mengambil artikel Wikipedia

Pipeline dimulai dari teks mentah. Buku memuat dump Wikipedia bahasa Inggris dengan satu baris `load_dataset`.

> **Listing buku: dataset yang sudah tidak bisa dimuat**
>
> ```python
> dataset = load_dataset("wikipedia", "20220301.en", split="train")
> ```
>
> Repositori `wikipedia` lama berisi skrip pemuat (`wikipedia.py`), dan `datasets` versi baru tidak lagi menjalankan skrip dataset: `RuntimeError: Dataset scripts are no longer supported, but found wikipedia.py`. Penggantinya `wikimedia/wikipedia`: isi dump yang sama, sudah berformat Parquet, dan bisa di-*stream*.

Dengan streaming, kita cukup membaca artikel dari awal dump sampai jumlah yang dibutuhkan, tanpa mengunduh 6,4 juta artikel. Artikel-artikel pertama dipakai untuk training, sisanya menjadi set held-out. Pemisahan ini menggantikan evaluasi buku yang, demi kesederhanaan, mengukur loss di data training. Sel ini hanya pekerjaan I/O dan CPU; GPU belum dipakai.

In [5]:
t0 = time.time()
wiki_stream = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
n_needed = CFG.n_train_articles + CFG.n_val_articles
articles = [
    {"id": a["id"], "title": a["title"], "text": a["text"]}
    for a in itertools.islice(wiki_stream, n_needed)
]
if len(articles) < n_needed:
    raise RuntimeError(f"hanya {len(articles)} artikel ter-stream, butuh {n_needed}")

train_articles = articles[:CFG.n_train_articles]
val_articles = articles[CFG.n_train_articles:]
assert not {a["id"] for a in train_articles} & {a["id"] for a in val_articles}

dataset = HFDataset.from_list([{"text": a["text"]} for a in train_articles])
val_dataset = HFDataset.from_list([{"text": a["text"]} for a in val_articles])

n_chars = [len(a["text"]) for a in train_articles]
print(f"{len(dataset)} artikel training + {len(val_dataset)} artikel held-out, di-stream dalam {time.time() - t0:.1f} s")
print(f"panjang artikel training: median {int(np.median(n_chars)):,} karakter, total {sum(n_chars) / 1e6:.1f} juta karakter")
print("tiga judul training pertama:", [a["title"] for a in train_articles[:3]])
print("judul held-out pertama     :", val_articles[0]["title"])

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

800 artikel training + 100 artikel held-out, di-stream dalam 7.6 s
panjang artikel training: median 16,513 karakter, total 18.4 juta karakter
tiga judul training pertama: ['Anarchism', 'Albedo', 'A']
judul held-out pertama     : Adam Weishaupt


Median panjang artikel memberi gambaran skala. Satu token GPT-2 rata-rata mewakili sekitar empat karakter teks bahasa Inggris, jadi konteks 512 token hanya memuat sekitar 2.000 karakter. Validasi lokal bagian data mencatat median 16.513 karakter untuk 800 artikel pertama dump, kira-kira delapan kali panjang satu jendela konteks. Jadi sebagian besar artikel tidak muat dalam satu jendela, dan fakta ini menentukan cara memotong teks di dua bagian berikutnya.

## 2. Preprocessing pertama: potong, pad, dan tandai padding

Model menerima ID token, bukan huruf. Listing pertama buku memakai tokenizer GPT-2 dengan `truncation=True` (potong di `max_length`) dan `padding="max_length"` (isi teks pendek sampai panjang yang sama), supaya setiap artikel menjadi satu baris tensor berukuran sama.

> **Listing buku: padding tanpa token padding**
>
> ```python
> tokenizer = AutoTokenizer.from_pretrained("gpt2")
> def preprocess_function(examples):
>     return tokenizer(examples["text"], truncation=True,
>                      max_length=512, padding="max_length")
> ```
>
> Tokenizer GPT-2 dilatih pada aliran teks kontinu dan tidak punya token PAD, sehingga `map` berhenti dengan `ValueError: Asking to pad but the tokenizer does not have a padding token`. Resep standarnya adalah `tokenizer.pad_token = tokenizer.eos_token`.

Tokenisasi berjalan di CPU; GPU baru dipakai saat training. Memakai EOS sebagai PAD memunculkan pertanyaan baru: bagaimana loss membedakan PAD dari EOS asli? Jawabannya `attention_mask`. Posisi dengan mask 0 adalah padding, dan di posisi itu label diganti **-100**, nilai yang dilewati oleh cross-entropy PyTorch maupun Hugging Face. Tanpa langkah ini, model ikut dilatih menebak deretan EOS di ujung teks pendek, tebakan yang mudah sekaligus tidak berguna.

In [6]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token          # GPT-2 tidak punya PAD; EOS dipinjam


def preprocess_function(examples):
    enc = tokenizer(examples["text"], truncation=True, max_length=SEQ_LEN, padding="max_length")
    # Label = input_ids, kecuali posisi padding (attention_mask 0) yang diberi -100.
    enc["labels"] = [
        [tok if keep == 1 else -100 for tok, keep in zip(ids, mask)]
        for ids, mask in zip(enc["input_ids"], enc["attention_mask"])
    ]
    return enc


tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
val_tokenized = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)
tokenized_dataset.set_format("torch")
val_tokenized.set_format("torch")

train_rows = tokenized_dataset[:]
train_mask = train_rows["attention_mask"]
assert (train_rows["labels"][train_mask == 0] == -100).all()
assert (train_rows["labels"][train_mask == 1] == train_rows["input_ids"][train_mask == 1]).all()

print(tokenized_dataset)
print(f"posisi padding: {(train_mask == 0).float().mean().item():.1%} dari {train_mask.numel():,} posisi")
print(f"artikel yang perlu padding: {(train_mask.sum(dim=1) < SEQ_LEN).sum().item()} dari {len(tokenized_dataset)}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 800
})
posisi padding: 4.6% dari 409,600 posisi
artikel yang perlu padding: 79 dari 800


Setiap baris kini berisi `input_ids`, `attention_mask`, dan `labels` sepanjang `SEQ_LEN`, dan dua asersi di sel itu memastikan aturan labelnya: -100 tepat di posisi padding, sama dengan `input_ids` di posisi lain. Persentase posisi padding menunjukkan seberapa banyak komputasi yang terbuang untuk mengisi kekosongan. Untuk artikel Wikipedia yang umumnya panjang, angkanya kecil: validasi lokal bagian data mencatat 4,6% posisi padding, seluruhnya berasal dari 79 artikel (dari 800) yang lebih pendek dari 512 token. Pemborosan yang lebih besar justru di arah sebaliknya: semua teks setelah token ke-512 dibuang oleh truncation. Bagian 4 mengukur seberapa banyak.

## 3. Loop training: forward, loss, backward, update

Inti pipeline adalah loop empat langkah yang diulang untuk setiap batch: **forward** (model memprediksi token berikutnya), **loss** (seberapa salah prediksinya), **backward** (menghitung gradien, yaitu arah perubahan setiap bobot yang menurunkan loss), dan **update** (optimizer menggeser bobot). *Scheduler* mengatur besar langkah update: naik bertahap selama *warmup*, lalu turun linear sampai nol.

> **Listing buku: loop yang berhenti di batch pertama**
>
> ```python
> train_dataloader = DataLoader(tokenized_dataset, shuffle=True, batch_size=8)
> device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
> for batch in train_dataloader:
>     batch = {k: v.to(device) for k, v in batch.items()}
>     outputs = model(batch)
>     loss = outputs.loss
> ```
>
> Dataset belum diubah ke format tensor, jadi `DataLoader` menghasilkan list berisi satu tensor per posisi token dan `.to(device)` gagal: `AttributeError: 'list' object has no attribute 'to'`. Di belakangnya menunggu dua masalah lain: `model(batch)` mengirim dict sebagai `input_ids` (yang dimaksud `model(**batch)`), dan tanpa `labels` model tidak menghitung loss sehingga `outputs.loss` bernilai `None`. Baris `device` juga diam-diam jatuh ke CPU bila GPU tidak ada; perannya kini diambil gate CUDA. Terakhir, `wandb.init(...)` di listing asli membutuhkan akun, jadi diganti log JSONL lokal.

Versi yang berjalan dibagi menjadi tiga sel: persiapan dengan sanity check, loop training, lalu checkpoint. Sebelum melatih, kita ukur dulu loss GPT-2 pretrained pada data held-out sebagai titik awal.

In [7]:
loader_gen = torch.Generator().manual_seed(SEED)
train_dataloader = DataLoader(tokenized_dataset, shuffle=True, batch_size=CFG.batch_size,
                              generator=loader_gen, pin_memory=PIN_MEMORY)
val_dataloader = DataLoader(val_tokenized, batch_size=CFG.eval_batch, pin_memory=PIN_MEMORY)


def load_model():
    if RUN_MODE == "gpu":
        return AutoModelForCausalLM.from_pretrained("gpt2").float()     # GPT-2 124M pretrained, seperti buku
    tiny = GPT2Config(vocab_size=len(tokenizer), n_positions=SEQ_LEN, n_embd=64, n_layer=2, n_head=2)
    return GPT2LMHeadModel(tiny)                                         # cpu_smoke: bobot acak


model = load_model().to(DEVICE)
model.config.pad_token_id = tokenizer.pad_token_id
param = next(model.parameters())
assert param.device.type == DEVICE.type
assert param.dtype == torch.float32

# Optimizer dibuat SETELAH model pindah device, jadi state Adam lahir di device yang sama.
# Default weight_decay torch.optim.AdamW = 0.01 (transformers.AdamW lama: 0.0).
optimizer = AdamW(model.parameters(), lr=CFG.lr)
num_training_steps = CFG.num_epochs * len(train_dataloader)
lr_scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=CFG.warmup_steps,
                                               num_training_steps=num_training_steps)

# Sanity check satu batch: device, shape, dan loss bawaan HF == loss token-weighted kita.
batch = to_device(next(iter(train_dataloader)))
model.eval()
with torch.no_grad():
    out = model(**batch)
    nll_sum, n_tok = causal_lm_nll(out.logits, batch["labels"])
model.train()
manual_loss = nll_sum / n_tok
print(f"model: {model.num_parameters():,} parameter | device bobot: {param.device} | device input: {batch['input_ids'].device}")
print(f"shape batch: {tuple(batch['input_ids'].shape)} | loss HF {out.loss.item():.4f} | loss manual {manual_loss.item():.4f}")
assert batch["input_ids"].device == param.device
assert torch.allclose(out.loss.float(), manual_loss, rtol=1e-4, atol=1e-4)
del out, batch
free_memory()

before = evaluate_lm(model, val_dataloader)
print(f"sebelum training : val loss {before['loss']:.3f} | perplexity {before['ppl']:.1f} | {before['tokens']:,} token held-out")
print(f"tebakan uniform  : ln({len(tokenizer)}) = {math.log(len(tokenizer)):.3f}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model: 124,439,808 parameter | device bobot: cuda:0 | device input: cuda:0
shape batch: (8, 512) | loss HF 3.4375 | loss manual 3.4375
sebelum training : val loss 3.198 | perplexity 24.5 | 49,680 token held-out
tebakan uniform  : ln(50257) = 10.825


Dua hal dicek sebelum langkah training pertama. Pertama, input dan bobot model sama-sama berada di GPU, dan optimizer dibuat **setelah** model dipindah sehingga state Adam juga lahir di GPU. Kedua, loss bawaan Hugging Face sama dengan hasil `causal_lm_nll`. Kesamaan ini menunjukkan bahwa model menggeser label di dalam: kita memberikan `labels` yang identik dengan `input_ids`, dan model sendiri yang memasangkan posisi *t* dengan token *t+1*.

Val loss "sebelum training" adalah kemampuan GPT-2 pretrained membaca Wikipedia. Pembandingnya ln(50257) ≈ 10,82, loss model yang menebak merata di seluruh 50.257 token kosakata. GPT-2 pretrained seharusnya jauh di bawah angka itu karena sudah belajar bahasa Inggris dari teks web.

In [8]:
model.train()
step = 0
t0 = time.time()
for epoch in range(CFG.num_epochs):
    for batch in train_dataloader:
        batch = to_device(batch)
        loss = model(**batch).loss          # input_ids, attention_mask, labels sebagai keyword
        require_finite(loss.detach(), f"loss langkah {step + 1}")
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        step += 1
        metrics_log.log(section="listing1", step=step, loss=loss.item(), lr=lr_scheduler.get_last_lr()[0])
        if step == 1 or step % 20 == 0 or step == num_training_steps:
            print(f"langkah {step:>4}/{num_training_steps} | loss {loss.item():.3f} | lr {lr_scheduler.get_last_lr()[0]:.2e}")
sync()
elapsed = time.time() - t0
print(f"{step} langkah dalam {elapsed:.1f} s ({elapsed / step * 1000:.0f} ms/langkah) | device {DEVICE}")
assert step == num_training_steps
assert lr_scheduler.get_last_lr()[0] == 0.0

after = evaluate_lm(model, val_dataloader)
print(f"val loss held-out: sebelum {before['loss']:.3f} -> sesudah {after['loss']:.3f} "
      f"| perplexity {before['ppl']:.1f} -> {after['ppl']:.1f}")

langkah    1/100 | loss 3.439 | lr 5.00e-06
langkah   20/100 | loss 3.478 | lr 4.44e-05
langkah   40/100 | loss 3.210 | lr 3.33e-05
langkah   60/100 | loss 3.216 | lr 2.22e-05
langkah   80/100 | loss 3.213 | lr 1.11e-05
langkah  100/100 | loss 3.267 | lr 0.00e+00
100 langkah dalam 126.4 s (1264 ms/langkah) | device cuda
val loss held-out: sebelum 3.198 -> sesudah 3.061 | perplexity 24.5 -> 21.3


Baca hasilnya dari dua sisi. Kolom `lr` menunjukkan scheduler bekerja: naik ke 5e-5 selama 10 langkah warmup, lalu turun linear sampai tepat 0 di langkah terakhir (asersi di akhir sel memastikannya). Loss training per batch naik-turun karena setiap batch berisi artikel yang berbeda. Angka yang lebih bisa dipercaya adalah perbandingan **val loss sebelum dan sesudah** pada artikel held-out yang sama.

Kalau val loss turun, model menjadi lebih baik memprediksi teks Wikipedia yang belum pernah dilihatnya, bukan sekadar menghafal batch training. Sebagai ilustrasi cara membaca (bukan hasil run): penurunan dari 3,40 ke 3,30 berarti perplexity turun dari e^3,40 ≈ 30 menjadi e^3,30 ≈ 27, sekitar 10% lebih sedikit "kebingungan" per token. Kalau val loss justru naik, curigai learning rate yang terlalu besar untuk fine-tuning atau data yang terlalu sedikit.

In [9]:
ckpt_dir = WORK_DIR / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / "listing1_epoch0.pt"
torch.save(
    {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": lr_scheduler.state_dict(),
        "step": step,
    },
    ckpt_path,
)

# Uji muat ulang: model baru dari config yang sama, bobot dari file, dibandingkan tensor per tensor.
state = torch.load(ckpt_path, map_location="cpu")
restored = AutoModelForCausalLM.from_config(model.config)
restored.load_state_dict(state["model"])
restored.to(DEVICE)
restored_sd = restored.state_dict()
identical = all(torch.equal(v, restored_sd[k]) for k, v in model.state_dict().items())
assert identical
assert state["step"] == step
print(f"checkpoint: {ckpt_path} ({ckpt_path.stat().st_size / 2**30:.2f} GiB) | bobot hasil muat ulang identik: {identical}")
del state, restored, restored_sd
free_memory()

checkpoint: /content/ch07_work/checkpoints/listing1_epoch0.pt (1.39 GiB) | bobot hasil muat ulang identik: True


Checkpoint menyimpan lebih dari bobot model. Untuk melanjutkan training persis di titik yang sama, kita juga butuh state optimizer (AdamW menyimpan dua rata-rata bergerak untuk setiap parameter) dan posisi scheduler. Karena itu ukuran file kira-kira tiga kali ukuran bobot: 124 juta parameter × 4 byte ≈ 0,5 GB untuk model, ditambah dua kali itu untuk state Adam. Uji muat ulang membandingkan setiap tensor bobot hasil muat dengan model asli, dan semuanya harus identik. Pembahasan checkpointing yang lebih lengkap ada di Bab 10 buku.

## 4. Data input lanjutan: dua sumber, label causal, dan batch seragam

Listing lanjutan buku menaikkan level pipeline data: menggabungkan beberapa sumber agar model melihat ragam bahasa yang lebih luas, membuat label causal LM secara eksplisit, dan menyusun batch dengan fungsi collate sendiri.

> **Listing buku: dua sumber yang sama-sama tidak bisa dimuat**
>
> ```python
> wiki_dataset = load_dataset("wikipedia", "20220301.en", split="train")
> books_dataset = load_dataset("bookcorpus", split="train")
> combined_dataset = concatenate_datasets([wiki_dataset, books_dataset])
> ```
>
> Masalahnya sama dengan bagian 1: `wikipedia` dan `bookcorpus` sama-sama dataset berbasis skrip, sehingga baris pertama sudah gagal dengan `RuntimeError` yang sama.

Ide penggabungannya tetap kita jalankan. Artikel training dibelah menjadi dua irisan yang tidak saling tumpang tindih, dan keduanya berperan sebagai dua "sumber". Dalam proyek nyata, irisan kedua diganti korpus lain yang lisensinya sesuai.

In [10]:
half = len(train_articles) // 2
wiki_part = HFDataset.from_list([{"text": a["text"]} for a in train_articles[:half]])
second_part = HFDataset.from_list([{"text": a["text"]} for a in train_articles[half:]])   # stand-in sumber kedua
combined_dataset = concatenate_datasets([wiki_part, second_part])
print(f"{len(wiki_part)} + {len(second_part)} -> {len(combined_dataset)} dokumen")

400 + 400 -> 800 dokumen


`concatenate_datasets` menyambung kedua tabel menjadi satu dataset. Langkah berikutnya di buku adalah preprocessing baru.

> **Listing buku: label digeser manual dan batch yang tidak seragam**
>
> ```python
> tokenized = tokenizer(examples["text"], truncation=True, max_length=1024)
> labels = [ids[1:] + [tokenizer.eos_token_id] for ids in input_ids]
> ...
> collate_fn=lambda x: {k: np.stack([xi[k] for xi in x]) for k in x[0]}
> ```
>
> Listing ini memotong tanpa padding, sehingga panjang setiap contoh berbeda dan `np.stack` gagal pada batch pertama (`ValueError: all input arrays must have the same shape`). Masalah kedua tidak memunculkan error sama sekali: baris `labels` menggeser target satu posisi, padahal model causal Hugging Face sudah menggeser label di dalam. Akibatnya, model dilatih menebak token **dua** langkah ke depan.

Bug kedua adalah jenis yang paling berbahaya: pipeline berjalan, loss turun, tetapi targetnya salah. Seberapa jauh salahnya bisa diukur dengan GPT-2 pretrained sebagai alat ukur.

### Under the hood 1: harga label yang digeser dua kali

Eksperimennya sederhana. Model yang sama (GPT-2 pretrained dengan bobot segar, mode evaluasi) membaca 16 artikel held-out yang sama dua kali. Yang berubah hanya labelnya: sekali `labels = input_ids` (benar), sekali label hasil pergeseran manual ala buku. Posisi padding diberi -100 di kedua versi, jadi jumlah token yang dihitung sama.

In [11]:
# === Under the hood 1: harga label yang digeser dua kali ===
probe_model = load_model().to(DEVICE).eval()
if RUN_MODE != "gpu":
    print("cpu_smoke: model mini acak, angka di bawah hanya menguji jalannya kode")

probe_ids = tokenizer([a["text"] for a in val_articles[:16]], truncation=True, max_length=SEQ_LEN)["input_ids"]
label_variants = {
    "labels = input_ids (benar)": probe_ids,
    "labels digeser manual (listing buku)": [ids[1:] + [tokenizer.eos_token_id] for ids in probe_ids],
}
for name, label_seqs in label_variants.items():
    res = evaluate_lm(probe_model, rows_loader(pad_rows(probe_ids, label_seqs), CFG.eval_batch))
    print(f"{name:<38}: loss {res['loss']:.3f} | perplexity {res['ppl']:>9.1f} | {res['tokens']:,} token")
print(f"{'tebakan uniform':<38}: loss {math.log(len(tokenizer)):.3f}")
del probe_model
free_memory()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

labels = input_ids (benar)            : loss 3.196 | perplexity      24.4 | 8,176 token
labels digeser manual (listing buku)  : loss 9.556 | perplexity   14126.9 | 8,176 token
tebakan uniform                       : loss 10.825


Loss dengan label yang benar adalah kemampuan normal GPT-2 membaca Wikipedia. Loss dengan label ala buku mengukur seberapa baik model yang sama menebak token dua langkah ke depan tanpa melihat token di antaranya. Tugas itu jauh lebih sulit, jadi loss-nya diperkirakan melonjak ke arah tebakan acak ln(50257) ≈ 10,82; selisih kedua baris di output adalah harga bug ini untuk run-mu. Sebagai pembanding historis, run notebook versi lama (CPU, konteks 96 token, 16 artikel) mencatat 3,29 untuk label benar dan 9,35 untuk label ala buku, atau perplexity sekitar 27 berbanding sekitar 11.500.

Model yang **dilatih** dengan label ini akan mengejar target yang hampir acak itu. Kurva loss-nya tetap turun, tetapi model yang dihasilkan memprediksi hal yang salah. Aturannya: untuk model causal Hugging Face, `labels = input_ids` (dengan -100 di posisi padding), dan biarkan model yang menggeser.

Kedua bug diselesaikan sekaligus oleh **packing**, resep standar pre-training. Semua artikel ditokenisasi tanpa dipotong, disambung menjadi satu aliran token dengan EOS sebagai pemisah antar-artikel, lalu dipotong menjadi blok berukuran tepat `SEQ_LEN`. Setiap blok penuh dengan token asli, jadi tidak ada padding, semua shape seragam, dan collate `np.stack` dari buku pun menjadi sah. Labelnya cukup salinan `input_ids`. Satu blok bisa memuat ujung satu artikel dan awal artikel berikutnya; EOS di antaranya memberi tahu model bahwa topik berganti. Seperti tokenisasi sebelumnya, packing adalah pekerjaan CPU.

In [12]:
def pack_function(examples):
    joined = []
    for ids in tokenizer(examples["text"])["input_ids"]:
        joined.extend(ids + [tokenizer.eos_token_id])
    n_blocks = len(joined) // SEQ_LEN
    blocks = [joined[i * SEQ_LEN:(i + 1) * SEQ_LEN] for i in range(n_blocks)]
    return {
        "input_ids": blocks,
        "attention_mask": [[1] * SEQ_LEN for _ in blocks],
        "labels": [list(b) for b in blocks],          # = input_ids; model yang menggeser
    }


def book_collate(examples):
    # Collate np.stack dari listing buku, kini sah karena semua blok sama panjang.
    return {k: np.stack([ex[k] for ex in examples]) for k in examples[0]}


packed_dataset = combined_dataset.map(pack_function, batched=True, remove_columns=combined_dataset.column_names)
val_packed = val_dataset.map(pack_function, batched=True, remove_columns=val_dataset.column_names)
val_packed = val_packed.select(range(min(CFG.val_blocks, len(val_packed))))

demo = book_collate([packed_dataset[i] for i in range(4)])
print("collate buku pada 4 blok packed:", {k: v.shape for k, v in demo.items()})

packed_dataset.set_format("torch")
val_packed.set_format("torch")
val_packed_loader = DataLoader(val_packed, batch_size=CFG.eval_batch, pin_memory=PIN_MEMORY)

packed_tokens = len(packed_dataset) * SEQ_LEN
truncated_tokens = int(tokenized_dataset[:]["attention_mask"].sum())
print(f"{len(combined_dataset)} dokumen -> {len(packed_dataset):,} blok packed | held-out: {len(val_packed)} blok")
print(f"token training: packing {packed_tokens:,} vs potong-dan-pad {truncated_tokens:,} "
      f"({packed_tokens / truncated_tokens:.1f}x)")

collate buku pada 4 blok packed: {'input_ids': (4, 512), 'attention_mask': (4, 512), 'labels': (4, 512)}
800 dokumen -> 8,129 blok packed | held-out: 128 blok
token training: packing 4,162,048 vs potong-dan-pad 390,602 (10.7x)


Perbandingan token menjelaskan alasan utama packing. Dengan potong-dan-pad, setiap artikel hanya menyumbang paling banyak `SEQ_LEN` token pertamanya, dan sisanya dibuang. Packing memakai seluruh isi artikel, sehingga dari korpus yang sama kita mendapat beberapa kali lipat token training. Pada validasi lokal bagian data, 800 artikel menghasilkan 8.129 blok packed, yaitu 4.162.048 token, dibanding 390.602 token asli dengan potong-dan-pad: sekitar 10,7 kali lipat. Dengan kata lain, sekitar 90% isi korpus tidak pernah dilihat model bila kita hanya memotong.

Mulai bagian ini, evaluasi memakai blok packed dari artikel held-out (`val_packed_loader`). Angkanya tidak bisa dibandingkan langsung dengan val loss di bagian 3: blok packed berisi seluruh isi artikel, sedangkan versi potong-dan-pad hanya bagian pembukanya.

## 5. Arsitektur: dari konfigurasi ke jumlah parameter

Buku mengelompokkan keputusan arsitektur menjadi tiga. **Kapasitas**: ukuran kosakata, panjang konteks, dimensi embedding, jumlah layer, dan jumlah attention head, yang bersama-sama menentukan jumlah parameter. **Rezim data**: ukuran dan kualitas dataset serta jumlah langkah training. Model yang jauh lebih besar dari datanya cenderung *overfit* (menghafal), sedangkan model yang terlalu kecil tidak mampu menyerap data yang melimpah. **Kendala**: compute, efisiensi arsitektur, dan algoritma training. *Scaling laws* menghubungkan ketiganya: performa naik secara terprediksi bila parameter, data, dan compute dinaikkan bersama secara proporsional. Karena biaya dan energi, arah riset belakangan lebih ke "lebih cerdas, bukan sekadar lebih besar", misalnya *mixture-of-experts* yang hanya mengaktifkan sebagian jaringan untuk setiap token.

Listing konfigurasi buku berjalan apa adanya. Sel di bawah membangunnya di device `meta`: PyTorch hanya mencatat shape setiap tensor tanpa mengalokasikan memori, jadi menghitung 124 juta parameter tidak butuh GPU maupun RAM besar.

In [13]:
# Hanya menghitung parameter: model dibangun di device "meta" (shape tanpa memori), jadi tidak butuh GPU.
def describe(cfg, label):
    with torch.device("meta"):
        m = GPT2LMHeadModel(cfg)
    n_params = m.num_parameters()
    n_emb = cfg.vocab_size * cfg.n_embd + cfg.n_positions * cfg.n_embd
    print(f"{label:<36}: {n_params:>12,} parameter | embedding {n_emb / n_params:.0%}")
    return n_params


config = GPT2Config(
    vocab_size=50257,   # GPT-2 vocabulary size
    n_positions=1024,   # Maximum sequence length
    n_ctx=1024,         # Context size
    n_embd=768,         # Embedding dimension
    n_layer=12,         # Number of transformer layers
    n_head=12,          # Number of attention heads
)
n_book = describe(config, "listing buku (12 layer, embedding 768)")
describe(GPT2Config(vocab_size=50257, n_positions=96, n_embd=128, n_layer=2, n_head=2),
         "GPT-2 mini (2 layer, embedding 128)")
n_ours = model.num_parameters()
print(f"{'model yang kita fine-tune':<36}: {n_ours:>12,} parameter",
      "(sama persis)" if n_ours == n_book else "(cpu_smoke: model mini)")

typo = GPT2Config(n_layers=24)
print("GPT2Config(n_layers=24) -> n_layer =", typo.n_layer, "| atribut tambahan n_layers =", getattr(typo, "n_layers", None))
print("n_ctx tersimpan sebagai atribut:", getattr(config, "n_ctx", None), "| yang dipakai model: n_positions =", config.n_positions)

listing buku (12 layer, embedding 768):  124,439,808 parameter | embedding 32%
GPT-2 mini (2 layer, embedding 128) :    6,841,984 parameter | embedding 94%
model yang kita fine-tune           :  124,439,808 parameter (sama persis)
GPT2Config(n_layers=24) -> n_layer = 12 | atribut tambahan n_layers = 24
n_ctx tersimpan sebagai atribut: 1024 | yang dipakai model: n_positions = 1024


Listing buku menghasilkan 124.439.808 parameter, angka "sekitar 124 juta" yang disebut buku. Di profil `gpu`, angka ini sama persis dengan model pretrained yang kita latih di bagian 3, jadi konfigurasi tersebut memang arsitektur GPT-2 small.

Output juga memperlihatkan dua hal lain. Pertama, embedding (tabel kosakata 50.257 × 768 ditambah tabel posisi 1.024 × 768) mengambil sekitar 32% parameter. Pada konfigurasi mini dengan embedding 128 dan 2 layer, porsi itu melonjak ke sekitar 94%: hampir seluruh kapasitas habis untuk merepresentasikan token sebelum satu layer transformer pun bekerja. Itulah sebabnya model riset kecil biasanya memakai kosakata yang lebih kecil. Kedua, `GPT2Config` menerima argumen yang tidak dikenalnya tanpa protes. `n_ctx` di listing buku tidak lagi dipakai (perannya diambil `n_positions`), dan salah ketik seperti `n_layers=24` hanya menjadi atribut tambahan, sementara jumlah layer tetap 12. Selalu periksa config hasilnya, bukan hanya argumen yang ditulis.

## 6. Loss dan optimasi: AdamW, warmup, dan gradient clipping

Loss yang dipakai sejak awal adalah **cross-entropy**: negatif log-probabilitas yang diberikan model kepada token yang benar. Kalau model memberi probabilitas 0,5 pada token yang benar, posisi itu menyumbang loss −ln 0,5 ≈ 0,69; probabilitas 0,01 menyumbang ≈ 4,6. Bagian ini menambahkan tiga alat yang menjaga training tetap stabil:

- **AdamW** dengan `weight_decay=0.01`: setiap langkah, bobot ditarik sedikit ke arah nol, terpisah dari gradien. Ini salah satu bentuk regularisasi.
- **Warmup lalu decay linear**: learning rate mulai dari nol, karena di langkah-langkah awal statistik Adam belum stabil dan update besar bisa merusak bobot pretrained.
- **Gradient clipping** `max_norm=1.0`: bila panjang (norm) vektor gradien gabungan melebihi 1, seluruh vektor diperkecil sampai panjangnya 1. Arahnya tetap, hanya besarnya yang dibatasi.

Listing buku di bagian ini membawa `model(batch)` yang sama seperti di bagian 3, ditambah `torch.tensor(v)` pada nilai yang sudah berupa tensor (berjalan, tetapi menyalin data dan memunculkan warning). Versi di bawah melanjutkan fine-tuning model yang sama di data packed dengan optimizer baru, sambil mencatat norm gradien **sebelum** dipotong.

In [14]:
# === Training lanjutan di data packed + Under the hood 2: seberapa sering clipping bekerja ===
del optimizer, lr_scheduler
free_memory()

before_opt = evaluate_lm(model, val_packed_loader)
opt_loader = DataLoader(packed_dataset, shuffle=True, batch_size=CFG.batch_size,
                        generator=loader_gen, pin_memory=PIN_MEMORY)
assert len(opt_loader) >= CFG.opt_steps
optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=CFG.warmup_steps,
                                            num_training_steps=CFG.opt_steps)

grad_norms = []
model.train()
t0 = time.time()
for step, batch in enumerate(itertools.islice(opt_loader, CFG.opt_steps), start=1):
    batch = to_device(batch)
    loss = model(**batch).loss
    require_finite(loss.detach(), f"loss langkah {step}")
    loss.backward()
    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # norm SEBELUM dipotong
    require_finite(total_norm, f"norm gradien langkah {step}")
    grad_norms.append(total_norm.item())
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    metrics_log.log(section="optimasi", step=step, loss=loss.item(), grad_norm=grad_norms[-1],
                    lr=scheduler.get_last_lr()[0])
    if step % 20 == 0 or step == CFG.opt_steps:
        print(f"langkah {step:>4} | loss {loss.item():.3f} | norm pra-clip {grad_norms[-1]:.2f} | "
              f"lr {scheduler.get_last_lr()[0]:.2e}")
sync()
print(f"{CFG.opt_steps} langkah dalam {time.time() - t0:.1f} s | device {DEVICE}")

norms = np.array(grad_norms)
print(f"norm gradien pra-clip: p50 {np.percentile(norms, 50):.2f} | p95 {np.percentile(norms, 95):.2f} | maks {norms.max():.2f}")
print(f"langkah yang terpotong (norm > 1.0): {(norms > 1.0).mean():.0%} dari {len(norms)}")
after_opt = evaluate_lm(model, val_packed_loader)
print(f"val loss held-out (packed): {before_opt['loss']:.3f} -> {after_opt['loss']:.3f}")

langkah   20 | loss 3.256 | norm pra-clip 3.17 | lr 4.44e-05
langkah   40 | loss 3.297 | norm pra-clip 3.04 | lr 3.33e-05
langkah   60 | loss 3.401 | norm pra-clip 2.84 | lr 2.22e-05
langkah   80 | loss 3.383 | norm pra-clip 2.91 | lr 1.11e-05
langkah  100 | loss 3.360 | norm pra-clip 2.93 | lr 0.00e+00
100 langkah dalam 127.3 s | device cuda
norm gradien pra-clip: p50 3.04 | p95 3.43 | maks 3.86
langkah yang terpotong (norm > 1.0): 100% dari 100
val loss held-out (packed): 3.356 -> 3.347


### Under the hood 2: seberapa sering clipping bekerja?

Tiga angka meringkas distribusi norm gradien. **p50** (median) adalah norm pada langkah yang tipikal, **p95** adalah batas bawah 5% langkah dengan norm terbesar, dan **maks** adalah langkah terburuk. Persentase "terpotong" menghitung langkah yang norm-nya di atas 1,0.

Kalau persentase terpotong kecil dan p50 jauh di bawah 1, clipping berperan sebagai pengaman untuk batch langka yang ekstrem. Kalau mayoritas langkah terpotong, clipping sudah menjadi bagian rutin dari setiap update. Dengan Adam, yang menormalkan besar gradien, efek utamanya adalah meredam pengaruh langkah-langkah besar terhadap rata-rata bergerak gradien, bukan sekadar mengecilkan langkah. Sebagai pembanding historis, run versi lama yang melatih model mini **dari nol** di CPU mencatat 53% langkah terpotong dengan p50 1,03. Fine-tuning model yang sudah matang bisa menghasilkan pola yang berbeda, dan output sel di atas menunjukkan pola mana yang terjadi di run-mu.

### Under the hood 3: batch size, throughput, dan memori

Buku menyebut batch size sebagai keseimbangan antara memori dan efisiensi. Eksperimen berikut mengukurnya langsung: untuk batch 1, 2, 4, dan 8 blok packed, jalankan forward dan backward tanpa update bobot, ukur waktu per langkah setelah satu langkah pemanasan, lalu catat memori GPU puncak. GPU di-*synchronize* sebelum stopwatch dibaca, karena perintah CUDA berjalan asinkron; tanpa sinkronisasi, waktu yang terukur hanya kecepatan mengirim perintah.

In [15]:
# === Under the hood 3: batch size vs throughput vs memori (float32, forward+backward, tanpa update) ===
del optimizer, scheduler
free_memory()
sweep_blocks = packed_dataset[:max(CFG.sweep_batches)]["input_ids"]
model.train()
print(f"{'batch':>5} | {'ms/langkah':>10} | {'token/detik':>12} | memori puncak")
for bs in CFG.sweep_batches:
    x = sweep_blocks[:bs].to(DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for i in range(4):                                  # langkah 0 = pemanasan, langkah 1-3 diukur
        if i == 1:
            sync()
            t0 = time.perf_counter()
        loss = model(input_ids=x, labels=x).loss
        require_finite(loss.detach(), f"loss sweep batch {bs}")
        loss.backward()
        model.zero_grad(set_to_none=True)
    sync()
    dt = (time.perf_counter() - t0) / 3
    print(f"{bs:>5} | {dt * 1000:>10.1f} | {bs * SEQ_LEN / dt:>12,.0f} | {peak_mem_text()}")
del x, loss
free_memory()

batch | ms/langkah |  token/detik | memori puncak
    1 |      156.7 |        3,268 | 1.33 GiB
    2 |      299.5 |        3,419 | 2.14 GiB
    4 |      583.3 |        3,511 | 3.80 GiB
    8 |     1203.3 |        3,404 | 7.11 GiB


Tiga kolom perlu dibaca bersamaan. **ms/langkah** naik seiring batch karena pekerjaannya bertambah. **token/detik** adalah throughput: kalau naik seiring batch, batch kecil belum menyibukkan seluruh core GPU, dan batch yang lebih besar memakai core yang sebelumnya menganggur. Saat kenaikannya melandai, GPU sudah jenuh. **Memori puncak** biasanya naik kurang lebih linear, karena aktivasi setiap contoh harus disimpan sampai backward selesai, sementara bobot dan gradien adalah porsi tetap yang tidak bergantung batch.

Bandingkan dengan run historis notebook versi lama di CPU satu core (model mini, konteks 96): di sana throughput justru tertinggi di batch 1, sekitar 1.550 token/detik, dan turun ke sekitar 1.050 token/detik di batch 16, karena satu core sudah jenuh sejak batch terkecil. Kalau tabel GPU-mu memperlihatkan throughput yang naik seiring batch, perbedaannya datang dari paralelisme hardware. Kesimpulan praktisnya bergantung hardware: di GPU, pilih batch terbesar yang masih muat dengan sisa memori yang aman. Bila batch yang diinginkan tidak muat, gradient accumulation di bagian 8 menjadi jalan keluarnya.

## 7. Logging: TensorBoard dan kelas `LLMTrainer`

Loop tanpa log adalah kotak hitam: kalau loss meledak di langkah ke-3.000, kita tidak tahu kapan mulainya. Buku memindahkan logging ke **TensorBoard**, yang menulis metrik ke file lokal, dan sekaligus membungkus loop ke dalam kelas `LLMTrainer`. Dengan kelas, logika training berada di satu tempat, bisa dipakai ulang untuk model atau data lain, dan mudah diperluas dengan evaluasi atau checkpoint.

> **Listing buku: log yang menyembunyikan learning rate**
>
> ```python
> self.writer.add_scalar('training_loss', cur_loss, global_step=i)
> print(f'| {i:5d}/{len(self.train_dataloader):5d} batches | '
>       f'lr {self.scheduler.get_last_lr()[0]:02.2f} | ...')
> ```
>
> Kode ini berjalan, tetapi lognya menyesatkan. Format `:02.2f` hanya menampilkan dua desimal, sehingga learning rate 5e-5 tercetak `0.00`. `global_step=i` dihitung ulang dari nol di setiap epoch, sehingga kurva epoch kedua menimpa kurva epoch pertama di TensorBoard. Kelas buku juga membawa `model(batch)` dan `torch.tensor(v)` dari bagian sebelumnya.

Sel kecil berikut, murni Python di CPU, memperlihatkan masalah format tersebut.

In [16]:
# Murni Python (CPU): bagaimana format log buku menampilkan learning rate.
for lr in (5e-5, 5e-4, 1e-2):
    print(f"lr sebenarnya {lr!r:>7} | format buku :02.2f -> {lr:02.2f} | format :.2e -> {lr:.2e}")

lr sebenarnya   5e-05 | format buku :02.2f -> 0.00 | format :.2e -> 5.00e-05
lr sebenarnya  0.0005 | format buku :02.2f -> 0.00 | format :.2e -> 5.00e-04
lr sebenarnya    0.01 | format buku :02.2f -> 0.01 | format :.2e -> 1.00e-02


Hanya 0.01 yang selamat dari format dua desimal; learning rate fine-tuning yang realistis lenyap menjadi `0.00`. Notasi ilmiah `:.2e` menampilkan semuanya.

Versi `LLMTrainer` di bawah mempertahankan struktur buku: konstruktor, `train_epoch`, dan `train`. Perbaikannya: `model(**batch)`, format `:.2e`, `global_step` yang terus naik lintas epoch, evaluasi held-out di setiap akhir epoch, pemeriksaan loss dan norm gradien yang finite, serta folder log unik per run sehingga run ulang tidak bercampur dengan run sebelumnya.

In [17]:
from torch.utils.tensorboard import SummaryWriter


class LLMTrainer:
    def __init__(self, model, train_dataloader, optimizer, scheduler, device,
                 val_dataloader=None, log_dir=None, log_interval=20, max_grad_norm=1.0):
        self.model = model
        self.train_dataloader = train_dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.val_dataloader = val_dataloader
        self.log_dir = Path(log_dir) if log_dir is not None else WORK_DIR / "runs" / f"trainer_{RUN_ID}"
        self.log_interval = log_interval
        self.max_grad_norm = max_grad_norm
        self.writer = SummaryWriter(log_dir=str(self.log_dir))
        self.global_step = 0                      # terus naik lintas epoch (buku memakai i per epoch)

    def _n_batches(self, max_batches):
        n = len(self.train_dataloader)
        return n if max_batches is None else min(n, max_batches)

    def _log_train(self, cur_loss, done, n_batches, ms_per_batch):
        lr = self.scheduler.get_last_lr()[0]
        self.writer.add_scalar("training_loss", cur_loss, global_step=self.global_step)
        self.writer.add_scalar("learning_rate", lr, global_step=self.global_step)
        print(f"| {done:5d}/{n_batches:5d} batches | global step {self.global_step:5d} | "
              f"lr {lr:.2e} | ms/batch {ms_per_batch:7.1f} | loss {cur_loss:5.2f}")

    def train_epoch(self, max_batches=None):
        self.model.train()
        n_batches = self._n_batches(max_batches)
        total_loss = 0.0
        n_logged = 0
        start_time = time.time()
        for i, batch in enumerate(itertools.islice(self.train_dataloader, n_batches)):
            batch = {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}
            loss = self.model(**batch).loss
            require_finite(loss.detach(), f"loss global step {self.global_step + 1}")
            total_loss += loss.item()
            n_logged += 1
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.max_grad_norm)
            require_finite(grad_norm, f"norm gradien global step {self.global_step + 1}")
            self.optimizer.step()
            self.scheduler.step()
            self.optimizer.zero_grad(set_to_none=True)
            self.global_step += 1
            if (i + 1) % self.log_interval == 0 or i + 1 == n_batches:
                ms_per_batch = (time.time() - start_time) * 1000 / n_logged
                self._log_train(total_loss / n_logged, i + 1, n_batches, ms_per_batch)
                total_loss = 0.0
                n_logged = 0
                start_time = time.time()

    def evaluate(self):
        result = evaluate_lm(self.model, self.val_dataloader)
        self.writer.add_scalar("val_loss", result["loss"], global_step=self.global_step)
        print(f"  evaluasi held-out @ global step {self.global_step}: "
              f"val loss {result['loss']:.3f} | perplexity {result['ppl']:.1f}")
        return result

    def train(self, num_epochs, max_batches_per_epoch=None):
        for epoch in range(num_epochs):
            print(f"Starting epoch {epoch + 1}")
            self.train_epoch(max_batches_per_epoch)
            if self.val_dataloader is not None:
                self.evaluate()
        self.writer.close()

Kita jalankan dua epoch pendek, masing-masing dibatasi `trainer_batches_per_epoch` batch, supaya kontinuitas `global_step` bisa dibuktikan. Setelah training selesai, file event TensorBoard dibaca ulang dan langkah-langkah `training_loss` di dalamnya harus terus naik tanpa duplikat.

In [18]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

free_memory()
trainer_loader = DataLoader(packed_dataset, shuffle=True, batch_size=CFG.batch_size,
                            generator=loader_gen, pin_memory=PIN_MEMORY)
n_epochs = 2
optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=min(5, CFG.warmup_steps),
                                            num_training_steps=n_epochs * CFG.trainer_batches_per_epoch)
trainer = LLMTrainer(model, trainer_loader, optimizer, scheduler, DEVICE,
                     val_dataloader=val_packed_loader,
                     log_dir=WORK_DIR / "runs" / f"llm_trainer_{RUN_ID}",
                     log_interval=CFG.trainer_log_interval)
trainer.train(num_epochs=n_epochs, max_batches_per_epoch=CFG.trainer_batches_per_epoch)

assert trainer.global_step == n_epochs * CFG.trainer_batches_per_epoch
assert scheduler.get_last_lr()[0] == 0.0

events = EventAccumulator(str(trainer.log_dir))
events.Reload()
logged_steps = [e.step for e in events.Scalars("training_loss")]
assert logged_steps == sorted(set(logged_steps)), logged_steps
print("global step training_loss di file event:", logged_steps)
print("global step val_loss di file event     :", [e.step for e in events.Scalars("val_loss")])

Starting epoch 1
|    10/   30 batches | global step    10 | lr 4.55e-05 | ms/batch  1232.1 | loss  3.40
|    20/   30 batches | global step    20 | lr 3.64e-05 | ms/batch  1271.2 | loss  3.42
|    30/   30 batches | global step    30 | lr 2.73e-05 | ms/batch  1288.2 | loss  3.38
  evaluasi held-out @ global step 30: val loss 3.352 | perplexity 28.5
Starting epoch 2
|    10/   30 batches | global step    40 | lr 1.82e-05 | ms/batch  1259.8 | loss  3.36
|    20/   30 batches | global step    50 | lr 9.09e-06 | ms/batch  1248.3 | loss  3.38
|    30/   30 batches | global step    60 | lr 0.00e+00 | ms/batch  1242.8 | loss  3.34
  evaluasi held-out @ global step 60: val loss 3.350 | perplexity 28.5
global step training_loss di file event: [10, 20, 30, 40, 50, 60]
global step val_loss di file event     : [30, 60]


Setiap baris log kini menampilkan learning rate yang sebenarnya, dan pembacaan ulang file event membuktikan bahwa langkah epoch kedua melanjutkan epoch pertama, bukan menimpanya. Folder `ch07_work/runs/` menyimpan log lengkapnya. Di Colab, kurvanya bisa dilihat dengan `%load_ext tensorboard` lalu `%tensorboard --logdir ch07_work/runs`; semuanya lokal dan tidak memerlukan akun.

## 8. Scaling: gradient accumulation dan mixed precision

Bila batch yang diinginkan tidak muat di memori GPU, buku memakai dua teknik.

**Gradient accumulation** memproses beberapa *micro-batch* kecil satu per satu, menjumlahkan gradiennya, lalu meng-update bobot sekali. Empat micro-batch berisi 4 blok memberi gradien setara satu batch 16 blok, dengan memori aktivasi seukuran batch 4.

**Mixed precision** menghitung forward dan backward dalam format 16-bit, sementara bobot master tetap 32-bit. Format 16-bit memakai separuh memori dan bisa berjalan di Tensor Core. Ada dua pilihan. `float16` punya rentang angka yang sempit, sehingga gradien yang sangat kecil bisa menjadi nol (*underflow*); `GradScaler` mengatasinya dengan mengalikan loss sebelum backward, lalu membagi kembali gradien sebelum update. `bfloat16` punya rentang yang sama dengan float32 sehingga tidak perlu scaler, tetapi baru didukung native mulai GPU generasi Ampere. Gate di awal notebook sudah memilih: T4 memakai float16 + GradScaler.

> **Listing buku: `LargeScaleLLMTrainer` yang belum pernah berjalan**
>
> ```python
> import torch.cuda.amp as amp
> with amp.autocast():
>     outputs = self.model(batch)
>     loss = outputs.loss / self.accumulation_steps
> if (i + 1) % self.accumulation_steps == 0:
>     ...
> elapsed = time.time() start_time
> ```
>
> Tanda minus di baris terakhir hilang, sehingga seluruh kelas gagal di-*parse* (`SyntaxError`). Di belakangnya: `torch.cuda.amp` sudah digantikan `torch.amp`, `model(batch)` muncul lagi, dan update hanya terjadi saat `i + 1` habis dibagi jumlah akumulasi. Kalau jumlah micro-batch dalam satu epoch tidak habis dibagi, gradien sisa di akhir epoch tidak pernah dipakai untuk update dan malah terbawa ke update pertama epoch berikutnya. Pembagian `loss / accumulation_steps` juga hanya tepat bila setiap micro-batch berisi jumlah token yang sama.

Versi di bawah memperbaiki semuanya. Micro-batch dikelompokkan menjadi **jendela** akumulasi, termasuk jendela terakhir yang tidak penuh. Loss setiap micro-batch dibagi jumlah token berlabel di seluruh jendela, sehingga gradien akhirnya sama dengan rata-rata per token satu batch besar. Clipping dilakukan setelah `unscale_`, jadi yang dipotong adalah gradien sebenarnya, bukan versi yang sudah dikalikan scaler. Bila GradScaler menemukan gradien inf/NaN, update itu dilewati dan scheduler ikut tidak dimajukan, agar jadwal learning rate tetap sejalan dengan update yang benar-benar terjadi.

In [19]:
class LargeScaleLLMTrainer(LLMTrainer):
    def __init__(self, model, train_dataloader, optimizer, scheduler, device,
                 accumulation_steps=4, amp_dtype=None, **kwargs):
        super().__init__(model, train_dataloader, optimizer, scheduler, device, **kwargs)
        self.accumulation_steps = accumulation_steps
        self.amp_dtype = amp_dtype
        # GradScaler hanya untuk float16; bfloat16 punya rentang eksponen float32.
        self.scaler = torch.amp.GradScaler(device.type, enabled=(amp_dtype == torch.float16))
        self.optimizer_steps = 0
        self.skipped_steps = 0
        self.window_sizes = []

    def _windows(self, batches):
        window = []
        for batch in batches:
            window.append(batch)
            if len(window) == self.accumulation_steps:
                yield window
                window = []
        if window:
            yield window                          # jendela terakhir yang tidak penuh tetap dipakai

    def train_epoch(self, max_batches=None):
        self.model.train()
        n_batches = self._n_batches(max_batches)
        nll_logged = 0.0
        tok_logged = 0
        micro_logged = 0
        done = 0
        start_time = time.time()
        for window in self._windows(itertools.islice(self.train_dataloader, n_batches)):
            # Penyebut = token berlabel di SELURUH jendela -> setara rata-rata per token satu batch besar.
            window_tokens = sum(int((b["labels"][:, 1:] != -100).sum()) for b in window)
            for batch in window:
                batch = {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}
                with torch.autocast(device_type=self.device.type, dtype=self.amp_dtype,
                                    enabled=self.amp_dtype is not None):
                    logits = self.model(input_ids=batch["input_ids"],
                                        attention_mask=batch["attention_mask"]).logits
                nll_sum, _ = causal_lm_nll(logits, batch["labels"])
                require_finite(nll_sum.detach(), f"loss micro-batch (update {self.optimizer_steps + 1})")
                self.scaler.scale(nll_sum / window_tokens).backward()
                nll_logged += nll_sum.item()
            tok_logged += window_tokens
            micro_logged += len(window)
            done += len(window)
            self.window_sizes.append(len(window))

            self.scaler.unscale_(self.optimizer)          # clipping pada gradien asli, bukan versi ter-scale
            grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.max_grad_norm)
            scale_before = self.scaler.get_scale()
            self.scaler.step(self.optimizer)              # float16: dilewati otomatis bila ada inf/NaN
            self.scaler.update()
            if self.scaler.is_enabled() and self.scaler.get_scale() < scale_before:
                self.skipped_steps += 1
            else:
                require_finite(grad_norm, f"norm gradien update {self.optimizer_steps + 1}")
                self.scheduler.step()
            self.optimizer.zero_grad(set_to_none=True)
            self.optimizer_steps += 1
            self.global_step += 1
            if self.global_step % self.log_interval == 0 or done == n_batches:
                ms_per_batch = (time.time() - start_time) * 1000 / micro_logged
                self._log_train(nll_logged / tok_logged, done, n_batches, ms_per_batch)
                nll_logged = 0.0
                tok_logged = 0
                micro_logged = 0
                start_time = time.time()

Untuk menguji penanganan jendela terakhir, data sengaja dibuat 30 micro-batch berisi 4 blok dengan akumulasi 4 (profil `gpu`). Harapannya 7 jendela penuh ditambah 1 jendela berisi 2 micro-batch, total 8 update. Jadwal learning rate juga dihitung dalam satuan update, bukan micro-batch.

In [20]:
free_memory()
micro_subset = packed_dataset.select(range(CFG.n_micro_batches * CFG.micro_batch))
micro_loader = DataLoader(micro_subset, shuffle=True, batch_size=CFG.micro_batch,
                          generator=loader_gen, pin_memory=PIN_MEMORY)
expected_updates = math.ceil(len(micro_loader) / CFG.accum_steps)

optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=2, num_training_steps=expected_updates)
trainer = LargeScaleLLMTrainer(
    model, micro_loader, optimizer, scheduler, DEVICE,
    accumulation_steps=CFG.accum_steps, amp_dtype=AMP_DTYPE,
    val_dataloader=val_packed_loader, log_dir=WORK_DIR / "runs" / f"large_scale_{RUN_ID}", log_interval=2,
)
trainer.train(num_epochs=1)

print(f"{len(micro_loader)} micro-batch x {CFG.micro_batch} blok, akumulasi {CFG.accum_steps} -> "
      f"{trainer.optimizer_steps} update (harapan {expected_updates})")
print(f"ukuran jendela: {trainer.window_sizes} | update dilewati GradScaler: {trainer.skipped_steps} | "
      f"dtype autocast: {AMP_DTYPE}")
assert trainer.optimizer_steps == expected_updates
assert sum(trainer.window_sizes) == len(micro_loader)

Starting epoch 1
|     8/   30 batches | global step     2 | lr 5.00e-05 | ms/batch   245.9 | loss  3.37
|    16/   30 batches | global step     4 | lr 3.33e-05 | ms/batch   231.7 | loss  3.27
|    24/   30 batches | global step     6 | lr 1.67e-05 | ms/batch   231.9 | loss  3.34
|    30/   30 batches | global step     8 | lr 0.00e+00 | ms/batch   235.5 | loss  3.38
  evaluasi held-out @ global step 8: val loss 3.358 | perplexity 28.7
30 micro-batch x 4 blok, akumulasi 4 -> 8 update (harapan 8)
ukuran jendela: [4, 4, 4, 4, 4, 4, 4, 2] | update dilewati GradScaler: 0 | dtype autocast: torch.float16


Output memastikan tiga hal. Jumlah update sama dengan harapan, artinya jendela terakhir tidak hilang, dan daftar ukuran jendela memperlihatkan jendela pendek itu di ujung. Kolom loss adalah rata-rata per token sepanjang jendela yang dilog. Jumlah update yang dilewati GradScaler memberi tahu apakah skala awalnya terlalu besar: beberapa lompatan di awal training float16 adalah perilaku normal saat scaler mencari skala yang aman, tetapi bila update terus-menerus dilewati, ada nilai yang benar-benar tidak stabil. Pada GPU bfloat16, scaler tidak aktif dan angkanya selalu 0.

### Under the hood 4: kapan akumulasi setara dengan satu batch besar?

Klaim "akumulasi menyimulasikan batch besar" bisa diuji langsung dengan membandingkan gradien. Kita hitung gradien satu batch penuh sebagai acuan, lalu gradien hasil akumulasi micro-batch dengan berbagai cara membagi loss. Ada dua ukuran: **cosine similarity** (1 berarti arah gradien identik) dan **rasio norm** (1 berarti besarnya sama). Dropout dimatikan (`model.eval()`) agar setiap forward deterministik, dan perkalian titik dihitung dalam float64 supaya galat pembulatan tidak ikut terbaca sebagai perbedaan.

Datanya dua jenis. Empat blok **packed** semuanya berisi `SEQ_LEN` token asli. Empat contoh **padded** sengaja dibuat berbeda panjang (di profil `gpu`: 512, 307, 153, dan 64 token), seperti batch dari data potong-dan-pad.

In [21]:
# === Under the hood 4: kapan akumulasi gradien setara dengan satu batch besar? ===
del optimizer, scheduler, trainer
free_memory()
model.eval()                                   # dropout mati: perbandingan butuh forward deterministik
ACC = 4


def accumulated_grads(micro_batches, divide_by, autocast_dtype=None, loss_scale=1.0):
    """Gradien hasil akumulasi. divide_by: "tokens" (bobot per token) atau angka (pembagi tetap ala buku)."""
    model.zero_grad(set_to_none=True)
    window_tokens = sum(int((b["labels"][:, 1:] != -100).sum()) for b in micro_batches)
    for b in micro_batches:
        b = to_device(b)
        with torch.autocast(device_type=DEVICE.type, dtype=autocast_dtype, enabled=autocast_dtype is not None):
            logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
        nll_sum, n_tok = causal_lm_nll(logits, b["labels"])
        if divide_by == "tokens":
            loss = nll_sum / window_tokens
        else:
            loss = (nll_sum / n_tok) / divide_by
        (loss * loss_scale).backward()
    return [p.grad.detach().clone() / loss_scale for p in model.parameters()]


def compare(reference, other):
    dot = sum((a.double() * b.double()).sum() for a, b in zip(reference, other))
    norm_ref = torch.sqrt(sum((a.double() ** 2).sum() for a in reference))
    norm_other = torch.sqrt(sum((b.double() ** 2).sum() for b in other))
    return (dot / (norm_ref * norm_other)).item(), (norm_other / norm_ref).item()


def split_rows(rows):
    return [{k: v[i:i + 1] for k, v in rows.items()} for i in range(rows["input_ids"].size(0))]


packed_rows = val_packed[:ACC]                                     # 4 blok, semuanya SEQ_LEN token asli
lengths = [SEQ_LEN, SEQ_LEN * 3 // 5, SEQ_LEN * 3 // 10, SEQ_LEN // 8]
long_ids = [ids for ids in tokenizer([a["text"] for a in val_articles])["input_ids"] if len(ids) >= SEQ_LEN][:ACC]
assert len(long_ids) == ACC
padded_seqs = [ids[:n] for ids, n in zip(long_ids, lengths)]
padded_rows = pad_rows(padded_seqs, padded_seqs)                   # 4 contoh, panjang berbeda

results = []
ref = accumulated_grads([packed_rows], "tokens")
results.append(("packed, dibagi ACC", *compare(ref, accumulated_grads(split_rows(packed_rows), ACC)), "≈1 | ≈1"))
results.append(("packed, lupa membagi ACC", *compare(ref, accumulated_grads(split_rows(packed_rows), 1)), "≈1 | ≈4"))
if AMP_DTYPE is not None:
    amp_scale = 1024.0 if AMP_DTYPE == torch.float16 else 1.0       # float16 butuh loss scaling, bfloat16 tidak
    amp_grads = accumulated_grads(split_rows(packed_rows), "tokens", AMP_DTYPE, amp_scale)
    results.append((f"packed, autocast {str(AMP_DTYPE).split('.')[-1]}, per token", *compare(ref, amp_grads), "≈1 (sedikit <1) | ≈1"))
    del amp_grads
del ref

ref = accumulated_grads([padded_rows], "tokens")
results.append(("padded, dibagi ACC", *compare(ref, accumulated_grads(split_rows(padded_rows), ACC)), "<1 | ≠1"))
results.append(("padded, bobot per token", *compare(ref, accumulated_grads(split_rows(padded_rows), "tokens")), "≈1 | ≈1"))
del ref

two_rows = {k: v[:2] for k, v in packed_rows.items()}             # jendela terakhir hanya 2 micro-batch
ref = accumulated_grads([two_rows], "tokens")
results.append(("jendela akhir 2 micro-batch, dibagi ACC", *compare(ref, accumulated_grads(split_rows(two_rows), ACC)), "≈1 | ≈0.5"))
del ref

model.zero_grad(set_to_none=True)
model.train()
free_memory()

print(f"{'varian (acuan: satu batch penuh, float32)':<44} | {'cosine':>9} | {'rasio norm':>10} | harapan (cos | rasio)")
for name, cos, ratio, expect in results:
    print(f"{name:<44} | {cos:>9.6f} | {ratio:>10.4f} | {expect}")
if AMP_DTYPE is None:
    print("(baris autocast dilewati: mode cpu_smoke tidak memakai mixed precision)")

varian (acuan: satu batch penuh, float32)    |    cosine | rasio norm | harapan (cos | rasio)
packed, dibagi ACC                           |  1.000000 |     1.0000 | ≈1 | ≈1
packed, lupa membagi ACC                     |  1.000000 |     4.0000 | ≈1 | ≈4
packed, autocast float16, per token          |  0.999847 |     1.0000 | ≈1 (sedikit <1) | ≈1
padded, dibagi ACC                           |  0.755652 |     1.3694 | <1 | ≠1
padded, bobot per token                      |  1.000000 |     1.0000 | ≈1 | ≈1
jendela akhir 2 micro-batch, dibagi ACC      |  1.000000 |     0.5000 | ≈1 | ≈0.5


Baris-baris tabel menjawab pertanyaan itu secara bertahap.

- **Packed, dibagi ACC**: cosine dan rasio norm seharusnya ≈ 1. Karena setiap micro-batch berisi jumlah token yang sama, rata-rata dari rata-rata sama dengan rata-rata keseluruhan. Akumulasi di sini bukan aproksimasi; selisih yang tersisa hanya galat pembulatan.
- **Lupa membagi ACC**: arahnya tetap (cosine ≈ 1), tetapi norm sekitar 4 kali lipat. Dengan SGD, efeknya sama dengan diam-diam mengalikan learning rate 4 kali. Adam menormalkan besar gradien sehingga efek langsungnya kecil, tetapi interaksinya dengan clipping berubah: norm yang 4 kali lebih besar membuat ambang `max_norm` lebih sering terlampaui.
- **Autocast** (hanya di GPU): cosine yang sedikit di bawah 1 dan rasio norm yang sedikit bergeser dari 1 adalah harga presisi 16-bit; pada enam desimal, cosine bisa saja masih tercetak 1,000000. Penyimpangan sekecil itu biasanya tenggelam oleh noise antar-batch, tetapi besarnya bergantung GPU dan dtype.
- **Padded, dibagi ACC**: kasus yang paling mudah lolos dari perhatian. Contoh 64 token mendapat bobot yang sama dengan contoh 512 token, sehingga gradiennya menyimpang dari batch penuh yang merata-ratakan per token.
- **Padded, bobot per token**: membagi dengan total token jendela mengembalikan kesetaraan. Cara inilah yang dipakai `LargeScaleLLMTrainer` di atas.
- **Jendela akhir 2 micro-batch, dibagi ACC**: rasio norm sekitar 0,5. Pembagi tetap `accumulation_steps` membuat update terakhir setiap epoch hanya separuh kekuatan.

Di atas dua teknik ini, buku menyebut tangga scaling berikutnya: *model parallelism* (membelah layer antar-GPU), *pipeline parallelism* (membelah model menjadi tahap-tahap berurutan), pustaka seperti DeepSpeed dan Megatron-LM untuk model miliaran parameter, serta *offloading* memori ke CPU atau NVMe yang menukar kapasitas dengan latensi transfer. Semuanya di luar jangkauan satu GPU Colab, tetapi masalah yang dipecahkannya sama dengan yang baru saja diukur: memori untuk aktivasi, bobot, dan state optimizer.

In [22]:
# Artefak run ini (semuanya lokal di WORK_DIR).
with open(metrics_log.path) as f:
    n_lines = sum(1 for _ in f)
print(f"log JSONL  : {metrics_log.path} ({n_lines} baris)")
print(f"TensorBoard: {WORK_DIR / 'runs'}")
print(f"checkpoint : {ckpt_path} ({ckpt_path.stat().st_size / 2**30:.2f} GiB)")

log JSONL  : /content/ch07_work/logs/metrics_20261029-032535.jsonl (200 baris)
TensorBoard: /content/ch07_work/runs
checkpoint : /content/ch07_work/checkpoints/listing1_epoch0.pt (1.39 GiB)


## Ringkasan

Pipeline lengkap kini berjalan di GPU. Wikipedia di-stream dan dipisah menjadi data training dan held-out, lalu ditokenisasi dengan dua cara: potong-dan-pad dengan label -100, kemudian packing. GPT-2 124M di-fine-tune dengan AdamW, warmup, dan clipping, dievaluasi pada artikel yang tidak pernah dilihat, disimpan sebagai checkpoint yang terbukti bisa dimuat ulang, dan dilog secara lokal ke JSONL dan TensorBoard.

Pelajaran terpenting bab ini ada pada label dan pembagi:

- Untuk model causal Hugging Face, `labels = input_ids`, dan model yang menggeser. Pergeseran manual membuat model mengejar target dua token ke depan; UTH-1 mengukur harganya (run historis CPU: loss 3,29 menjadi 9,35, mendekati tebakan acak 10,82).
- Posisi padding diberi label -100, karena pada GPT-2 PAD = EOS dan hanya `attention_mask` yang bisa membedakan keduanya.
- Akumulasi gradien setara batch besar hanya bila pembaginya tepat: total token di jendela, termasuk jendela terakhir yang tidak penuh (UTH-4).
- Jadwal learning rate dihitung dalam satuan update optimizer, dan format log harus mampu menampilkan angkanya.
- Batch size dan clipping adalah keputusan yang bergantung hardware dan rezim training. UTH-2 dan UTH-3 memberi cara membacanya dari angka run-mu sendiri.

**Bab berikutnya:** setelah pipeline berjalan, pertanyaan selanjutnya adalah nilai knob mana yang terbaik. Bab 8 membahas *hyperparameter tuning*.